# LakeLogic — 5 Minute Quickstart

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LakeLogic/LakeLogic/blob/main/examples/colab/00_quickstart.ipynb)

One contract. One pipeline. Every row accounted for. Five minutes.

In [ ]:
import subprocess, sys, importlib, urllib.request, os
if importlib.util.find_spec("lakelogic") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "lakelogic[polars]"])
if not os.path.exists("_setup.py"):
    urllib.request.urlretrieve("https://raw.githubusercontent.com/LakeLogic/LakeLogic/main/examples/colab/_setup.py", "_setup.py")
from _setup import *

## The Problem

You have raw order data landing in your lake. Some rows have bad emails, negative amounts, or unknown statuses. You need to validate every row, quarantine the bad ones, and prove nothing was silently dropped — with zero custom Python logic.

## The Solution

In [ ]:
contract = write_contract("""
version: 1.0.0
dataset: orders

info:
  title: E-Commerce Orders
  version: 1.0.0
  owner: data-team@company.com
  target_layer: silver

model:
  fields:
    - name: order_id
      type: integer
      required: true
    - name: customer_email
      type: string
      required: true
      pii: true
    - name: amount
      type: float
      required: true
    - name: currency
      type: string
    - name: status
      type: string
    - name: created_at
      type: string

transformations:
  - phase: "post"
    derive:
      field: "amount_gbp"
      sql: "CAST(CASE WHEN currency='USD' THEN amount*0.79 WHEN currency='EUR' THEN amount*0.86 ELSE amount END AS DECIMAL(10,2))"

quality:
  row_rules:
    - name: valid_email
      sql: "customer_email LIKE '%@%.%'"
    - name: positive_amount
      sql: "amount > 0"
    - name: valid_status
      sql: "status IN ('pending','shipped','delivered','returned')"
    - name: valid_currency
      sql: "currency IN ('GBP','USD','EUR')"
    - name: valid_order_id
      sql: "order_id > 0"
      
""", "orders_contract.yaml")

# Generate 1000 rows — 10% intentionally bad
source_df = DataGenerator(contract).generate(rows=1000, invalid_ratio=0.10)

# Run the pipeline
proc = DataProcessor(contract, engine="polars")
good, bad = proc.run(source_df)

## The Proof

In [ ]:
# Every row accounted for
assert_reconciliation(source_df, good, bad)

In [ ]:
# What was caught
print("Quarantined rows (sample):")
display(bad.head(10))

In [ ]:
# What was good
print("Valid rows (sample):")
display(good.head(10))

In [ ]:
# Full audit trail
print_report(proc)

## What You Just Saw

- **One YAML contract** defined schema, quality rules, and a derived column
- **100% reconciliation** — source == good + bad, every row accounted for
- **Automatic audit trail** — run ID, timestamp, per-rule failure counts

---
## Go Deeper — Explore by Capability

Each notebook below maps to a pillar of LakeLogic's [Technical Capabilities](https://lakelogic.github.io/LakeLogic/#technical-capabilities):

| # | Notebook | What You'll See |
|---|---|---|
| 🛡️ | **[Data Quality & Trust](01_data_quality_trust.ipynb)** | Reconciliation proofs, Pydantic validation, SQL-first rules, SLO monitoring |
| 📜 | **[Compliance & Governance](02_compliance_governance.ipynb)** | GDPR erasure in 2 lines, automatic lineage, cost intelligence |
| ⚡ | **[Engine & Scale](03_engine_scale.ipynb)** | Same contract on Polars & DuckDB, incremental processing, dry run |
| 🔌 | **[Integrations](06_integrations.ipynb)** | dbt adapter, dlt sources, contract-driven quality gates on arrival |

> **Each notebook is self-contained** — pick the capability that matters most to you and run it independently.
